# 🧭 From finding structure to predicting answers

**Module 1 · Session 09, part 1**

For three sessions you have worked with one table: 780 cities from Nomad List, with columns for
cost, internet, safety, freedom, nightlife and so on. You compressed it with PCA, grouped it with
k-means, found its extremes with archetypal analysis, and searched it for similar cities.

All of that looked only at the columns. Nobody told the algorithm what the right answer was,
because there wasn't one. You judged the results by whether they made sense.

Today we hide one column and try to predict it. That is supervised learning, and the one thing
it adds is an answer key. Everything else in this notebook follows from being able to check.

**The brief.** A relocation agency helps remote workers choose a city. New cities appear on
Nomad List with ratings for internet, safety and so on, often before anyone has reported what
it costs to live there. The agency wants an estimate of monthly cost for those cities, and a
flag for the ones likely to be expensive.

In [ ]:
# Setup. One style block so every chart here matches the earlier sessions.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             mean_absolute_error, precision_score, r2_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8b8a85"
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE, GREY],
              rc={"figure.dpi": 110, "axes.titleweight": "bold"})

In [ ]:
CITIES = "https://sds-aau.github.io/SDS-master/M1/data/cities.csv"
raw = pd.read_csv(CITIES)
print(raw.shape)
raw.head(3)

Same file as sessions 06 and 07. Four text columns (the city and where it is), then 21 numbers.
`cost_nomad` is the estimated monthly cost of living for a remote worker, in US dollars.

## 🔎 0. One look before we trust it

Session 03 rule: before any model, look for values that cannot be right. Most columns are
scores between 0 and 1, so the maximum of each column is a quick check.

In [ ]:
raw.describe().T[["min", "50%", "max"]]

Two things are wrong.

- `racism` has a maximum of about 1.8 × 10²⁸ on a 0-to-1 scale. That is one city.
- The tiny negative minimums (around -5 × 10⁻¹⁷) are floating-point dust, zero in practice.
  They are harmless.

In [ ]:
raw.loc[raw["racism"] > 1, ["place", "alpha-2", "racism"]]

Samara. The value is a scraping error, not a measurement, so we replace it with the median
of the column. Say the decision out loud: one city, one column, median fill. Deleting the
whole city would also be defensible.

Section 5 shows what this single value does to a model if you leave it in.

In [ ]:
cities = raw.copy()
cities.loc[cities["racism"] > 1, "racism"] = cities["racism"].median()

# And one more oddity, for the record:
print("coffee equals beer in every row:", (cities["coffee_in_cafe"] == cities["cost_beer"]).all())

Every city in the file has a coffee that costs exactly as much as a beer. Another scraping
artefact. It will matter in section 6.

## 🔁 1. What you did last week, in three cells

Scale, compress, cluster. Nothing new here, just a reminder that it was all built on `X` alone.

In [ ]:
numeric = cities.select_dtypes("number")
Z = StandardScaler().fit_transform(numeric)

pca = PCA(n_components=2).fit(Z)
print("variance explained by two components:", pca.explained_variance_ratio_.round(2))

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
cities["cluster"] = kmeans.labels_
cities["cluster"].value_counts().sort_index()

## 🙈 2. Hide one column

Now the new move. We take `cost_nomad` out of the table and make it the thing to predict: `y`.
The other columns are the features: `X`.

We also set aside four more columns, the other prices. Section 6 explains why. Until then,
take it on trust that a city with no cost-of-living report probably has no coworking price
either.

In [ ]:
OTHER_PRICES = ["cost_coworking", "cost_expat", "coffee_in_cafe", "cost_beer"]
FEATURES = [c for c in numeric.columns if c not in ["cost_nomad", *OTHER_PRICES]]

X = cities[FEATURES]
y = cities["cost_nomad"]
print(len(FEATURES), "features:", FEATURES)

## 🏘️ 3. You already built a supervised model

Session 08: find the cities most like a given one. Here is Aalborg, on the scaled features.

In [ ]:
Zx = StandardScaler().fit_transform(X)
finder = NearestNeighbors(n_neighbors=6).fit(Zx)

aalborg = cities.index[cities["place"] == "Aalborg"][0]
_, idx = finder.kneighbors(Zx[[aalborg]])
neighbours = [i for i in idx[0] if i != aalborg][:5]   # the nearest city is Aalborg itself

cities.loc[neighbours, ["place", "alpha-2", "cost_nomad"]]

In [ ]:
guess = cities.loc[neighbours, "cost_nomad"].mean()
truth = cities.loc[aalborg, "cost_nomad"]
print(f"average of the neighbours: {guess:,.0f}")
print(f"Aalborg, actual:           {truth:,.0f}")

That average is a prediction. Averaging the neighbours' `y` is **k-nearest-neighbours
regression**, and scikit-learn has it as one object. Train it on every city except Aalborg,
then ask it about Aalborg:

In [ ]:
others = cities.index != aalborg
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))
knn.fit(X[others], y[others])
knn.predict(X.loc[[aalborg]])

The same number. `fit` stored the other cities and `predict` looked up the neighbours and
averaged them. Almost the same code as session 08, with a `y`.

Is being off by about 840 dollars good? We cannot say without something to compare it to.
Section 4 builds that.

## 🧩 Clusters are not labels

The clusters from section 1 were built on all the columns, including cost. So do they answer
the question "is this city expensive"?

In [ ]:
expensive = cities["cost_nomad"] > 3000
pd.crosstab(cities["cluster"], expensive, normalize="index").round(2)

Some clusters are almost all cheap, others are mixed. Clustering found structure that is
*related* to cost, but it was never asked about cost, and one cluster is roughly a coin flip.

If cost is what you care about, ask about cost. Supervision means exactly that.

## 📏 4. A baseline and an honest test

Two habits come before any real model:

1. **Hold some data back.** The model trains on 80% of the cities. The other 20% are the test set,
   standing in for the next cities that show up.
2. **Know what doing nothing scores.** The dumbest reasonable regression predicts the training
   average for every city.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
print(f"baseline MAE: {mean_absolute_error(y_test, baseline.predict(X_test)):,.0f} USD")

Predicting the average is off by about 900 dollars a month per city. Every model below has to
beat that to be worth anything.

For Aalborg, the average (about 2,330) would have been off by almost 1,900. The neighbours did
much better on that one city. One city proves nothing, which is why we use a test set.

## 🏋️ 5. Learning versus memorising

Five models, each scored on the rows it trained on and on the rows it never saw.

In [ ]:
models = {
    "1-nearest neighbour":           make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=1)),
    "decision tree, no depth limit": DecisionTreeRegressor(random_state=0),
    "linear regression":             make_pipeline(StandardScaler(), LinearRegression()),
    "decision tree, depth 3":        DecisionTreeRegressor(max_depth=3, random_state=0),
    "random forest":                 RandomForestRegressor(n_estimators=300, random_state=0),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append({"model": name,
                 "train MAE": mean_absolute_error(y_train, model.predict(X_train)),
                 "test MAE": mean_absolute_error(y_test, model.predict(X_test))})

pd.DataFrame(rows).set_index("model").round(0)

Read the two columns separately.

The nearest-neighbour model and the unlimited tree are nearly perfect on the training rows, and
among the worst on the test rows. They did not learn how cost relates to the features. They
memorised the training cities. If you ranked these models by training error, you would pick the
worst one.

The random forest is the best on test, even though it too is much better on train. That gap is
normal for forests: each tree fits its training rows closely, and the averaging happens across
trees, not within them. The test column is the one to read.

### The complexity knob

For k-nearest neighbours the knob is `k`. With `k = 1` the model copies the closest city. With
`k` equal to all the cities, it predicts the average. Somewhere between them is the best trade.

In [ ]:
ks = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100, 200, 400]
curve = []
for k in ks:
    m = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k)).fit(X_train, y_train)
    curve.append({"k": k,
                  "train": mean_absolute_error(y_train, m.predict(X_train)),
                  "test": mean_absolute_error(y_test, m.predict(X_test))})
curve = pd.DataFrame(curve)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["k"], curve["train"], marker="o", color=BLUE, label="train")
ax.plot(curve["k"], curve["test"], marker="o", color=ORANGE, label="test")
ax.set(xscale="log", xlabel="k (log scale)   ←  more flexible      simpler  →",
       ylabel="MAE (USD)", title="k-nearest neighbours: error against k")
ax.legend()
plt.show()

On the left, variance: training error near zero, test error high. On the right, bias: both
errors climb towards the baseline. The test curve is lowest somewhere in between.

That is the bias-variance tradeoff, measured on the data rather than drawn as a textbook
curve. It is also noisy: with 156 test cities, the exact bottom moves if you change the split.
That noise is the reason for the next section.

## 🔁 Cross-validation

One test split is one roll of the dice. Cross-validation splits the *training* data into five
folds, trains on four, scores the fifth, and rotates. Five scores give you an average and a
spread.

In [ ]:
for name in ["linear regression", "random forest"]:
    scores = -cross_val_score(models[name], X_train, y_train, cv=5,
                              scoring="neg_mean_absolute_error")
    print(f"{name:18s} folds: {scores.round(0)}   mean {scores.mean():,.0f}   sd {scores.std():,.0f}")

The forest beats linear regression on the mean, and its spread across folds is similar.

Cross-validation gives you a better estimate. It does not by itself make a model overfit less.
It helps with overfitting when you use it to *choose* settings, as in section 9.

### What Samara would have done

Here is the same linear regression run on the raw file, before the fix in section 0.

In [ ]:
X_raw_train, _, y_raw_train, _ = train_test_split(raw[FEATURES], raw["cost_nomad"],
                                                  test_size=0.2, random_state=42)
scores = -cross_val_score(make_pipeline(StandardScaler(), LinearRegression()),
                          X_raw_train, y_raw_train, cv=5, scoring="neg_mean_absolute_error")
print("folds:", [f"{s:.3g}" for s in scores])

One fold has an error of about 10²⁸ dollars. Samara is in that fold's validation rows, and a
linear model multiplies its absurd `racism` value by a coefficient. The other four folds look
normal.

When one fold is wildly worse than the rest, suspect the data before the model.

## 🚰 6. Leakage: what will you know when you predict?

Now the four columns set aside in section 2. Here is linear regression with and without them.

In [ ]:
def linear_r2(columns):
    Xa, Xb, ya, yb = train_test_split(cities[columns], y, test_size=0.2, random_state=42)
    model = make_pipeline(StandardScaler(), LinearRegression()).fit(Xa, ya)
    return r2_score(yb, model.predict(Xb))

print(f"with the other prices:    R² = {linear_r2(FEATURES + OTHER_PRICES):.2f}")
print(f"without the other prices: R² = {linear_r2(FEATURES):.2f}")

With the other prices the model looks twice as good. Now go back to the brief: a city where
nobody has reported the cost of living yet. Will the agency know what a coworking desk costs
there? What an expat reports spending? Almost certainly not. Those numbers come from the same
reports as `cost_nomad`.

A feature is only legitimate if it exists **at the moment the prediction is made**. When it
doesn't, you have leakage. Leaky models look excellent in the notebook and fail in use.

(The coffee and beer columns, which are identical, are one piece of leakage counted twice.)

## 🏷️ 7. Classification: is this city expensive?

The same features, a different kind of `y`: 1 if the city costs more than 3,000 USD a month,
otherwise 0.

In [ ]:
y_class = (cities["cost_nomad"] > 3000).astype(int)
print(f"share expensive: {y_class.mean():.2f}")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class)   # keep 28% in both halves

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_train, yc_train)
always_no = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)

print(f"logistic regression accuracy:  {accuracy_score(yc_test, logit.predict(Xc_test)):.2f}")
print(f"'never expensive' accuracy:    {accuracy_score(yc_test, always_no.predict(Xc_test)):.2f}")

A model that never says "expensive" is right 72% of the time, because 72% of cities are not
expensive. Our model is only a few points better. Accuracy hides almost everything when one
class is much bigger than the other, so look inside.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_estimator(logit, Xc_test, yc_test, display_labels=["cheap", "expensive"],
                                      cmap="Blues", colorbar=False, ax=ax)
ax.set_title("threshold 0.5")
plt.show()

pred = logit.predict(Xc_test)
print(f"precision: {precision_score(yc_test, pred):.2f}   when it says expensive, how often is it right?")
print(f"recall:    {recall_score(yc_test, pred):.2f}   of the expensive cities, how many did it find?")

It finds about a third of the expensive cities. For an agency warning clients about budget,
that is most of the job missed.

### The threshold is a choice

`predict` turns probabilities into labels with a cut-off of 0.5. `predict_proba` gives the
probabilities themselves, and you choose the cut-off.

In [ ]:
proba = logit.predict_proba(Xc_test)[:, 1]

rows = []
for t in np.arange(0.1, 0.91, 0.05):
    p = (proba >= t).astype(int)
    rows.append({"threshold": t, "precision": precision_score(yc_test, p, zero_division=0),
                 "recall": recall_score(yc_test, p), "accuracy": accuracy_score(yc_test, p)})
sweep = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4))
for col, colour in [("precision", BLUE), ("recall", ORANGE), ("accuracy", GREY)]:
    ax.plot(sweep["threshold"], sweep[col], marker="o", color=colour, label=col)
ax.axvline(0.5, color=GREY, linestyle=":")
ax.set(xlabel="threshold", ylabel="score", title="Same model, different cut-offs")
ax.legend()
plt.show()

print(f"ROC-AUC: {roc_auc_score(yc_test, proba):.2f}")
sweep[sweep["threshold"].round(2).isin([0.3, 0.5])].round(2)

Recall and precision move in opposite directions while accuracy stays almost flat. Lowering the
cut-off to 0.3 more than doubles recall, and accuracy hardly notices.

ROC-AUC summarises how well the probabilities *rank* expensive cities above cheap ones across
all thresholds. Choosing the threshold depends on what each mistake costs. That is session 10.

## 🌳 8. What is the model doing?

A shallow tree can be drawn and read.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y_class)

fig, ax = plt.subplots(figsize=(11, 4.5))
plot_tree(tree, feature_names=FEATURES, class_names=["cheap", "expensive"],
          filled=True, rounded=True, impurity=False, proportion=True, ax=ax)
plt.show()

The tree asks first about the fragile states index, then about press freedom (on that index,
a lower number means a freer press). Stable countries with a free press are mostly expensive.
Fragile states are mostly cheap.

For bigger models, two common summaries:

In [ ]:
forest = models["random forest"]
importance = pd.Series(forest.feature_importances_, index=FEATURES).sort_values()

linear = models["linear regression"]
coefficients = pd.Series(linear[-1].coef_, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
importance.plot.barh(ax=axes[0], color=BLUE, title="Random forest: feature importance")
coefficients.plot.barh(ax=axes[1], color=[ORANGE if v < 0 else BLUE for v in coefficients],
                       title="Linear regression: USD per 1 SD of feature")
plt.tight_layout()
plt.show()

print("correlation of life_score with cost:", round(cities["life_score"].corr(y), 2))

Both give the fragile states index the biggest role. Now look at `life_score` on the right: its
coefficient is strongly negative, even though on its own it correlates *positively* with cost.
(`freedom_score` does the same.)

That is not a bug. A coefficient is the effect of a feature **holding the others fixed**, and
`life_score` overlaps heavily with freedom, peace and stability. Once those are in the model,
what is left of `life_score` points the other way.

Two lessons:

- Coefficients, importances and SHAP values (part 2) describe **the model**, not the world.
- A feature that predicts cost does not cause cost. Stable, rich countries are expensive, and
  the fragile states index is a marker of that, not a lever.

## 🧪 9. Pipelines and tuning

Two last habits.

**Preprocessing goes inside the pipeline.** Every model above that needed scaling had its
`StandardScaler` inside `make_pipeline`. When the pipeline is fitted on the training rows, the
scaler learns means and standard deviations from those rows only. If you scale the whole table
first, the test rows leak into training.

**Choose settings with cross-validation, and touch the test set once.** `GridSearchCV` tries each
value with 5-fold cross-validation on the training data only.

In [ ]:
pipe = make_pipeline(StandardScaler(), KNeighborsRegressor())
search = GridSearchCV(pipe, {"kneighborsregressor__n_neighbors": [1, 3, 5, 10, 20, 50]},
                      cv=5, scoring="neg_mean_absolute_error")
search.fit(X_train, y_train)

print("best k:", search.best_params_["kneighborsregressor__n_neighbors"])
print(f"cross-validated MAE: {-search.best_score_:,.0f}")
print(f"test MAE, checked once at the end: {mean_absolute_error(y_test, search.predict(X_test)):,.0f}")

The test error is a little worse than the cross-validated one. That is normal: the chosen `k`
was the one that looked best on these folds, so its score is slightly optimistic.

### Components as features

One last bridge from session 06: PCA scores can be the input to a supervised model.

In [ ]:
for n in [2, 5, 10]:
    m = make_pipeline(StandardScaler(), PCA(n_components=n), LinearRegression()).fit(X_train, y_train)
    print(f"{n:2d} components -> test MAE {mean_absolute_error(y_test, m.predict(X_test)):,.0f}")

Two components already get close to linear regression on all sixteen features, and ten do a
little better. The first component is largely a stability-and-wealth axis, and that is also what
drives cost here, so compressing loses little. That is luck, not a rule: PCA picks the directions
with the most *variance* in X without looking at `y`, and on another problem the direction you
need can be one it discards.

## 📋 Where that leaves the brief

- Predicting a new city's cost from its ratings works, up to a point. The best model here is
  off by about 550 dollars a month, against 900 for guessing the average.
- The other price columns would make the model look far better, but the agency will not have
  them for a new city.
- For the expensive flag, the default cut-off misses most expensive cities. The cut-off should
  come from what a miss costs the agency, compared with a false alarm.
- The strongest predictors describe stable, wealthy countries. They are markers, not causes.

Part 2 applies the same workflow to messier data: Copenhagen Airbnb listings, with feature
engineering, gradient boosting and SHAP.

---

# ✏️ Exercises

Try each one before opening the solutions below.

**1. Your own city.** Pick another city in the file (Copenhagen, Lisbon or Chiang Mai are
interesting). Find its five nearest neighbours on `FEATURES`, predict its cost from them, and
compare the error with what the baseline would have been for that city. Is your city one where
similarity works?

*Hint: reuse `Zx`, `finder` and the section 3 code, and change the city name.*

**2. A tree's complexity knob.** Do for `max_depth` of a `DecisionTreeRegressor` what section 5
did for `k`: train and test MAE for depths 1 to 12 and no limit. Where is the test error lowest?

*Hint: `None` means no limit. Plot `range(1, 13)` and add the unlimited tree as a printed number.*

**3. Tune it properly.** Use `GridSearchCV` to choose `max_depth` for the tree with 5-fold
cross-validation. Does the tuned tree beat the random forest on the test set?

**4. A threshold from costs.** Say the agency loses 500 USD of goodwill for every expensive city
it fails to flag, and 100 USD of analyst time for every false alarm. Which threshold minimises
the total cost?

*Hint: choose the threshold on the training data, not the test set. `cross_val_predict(...,
method="predict_proba")` gives an out-of-fold probability for every training city.*

---

# ✅ Solutions

In [ ]:
# 1. Your own city
city = "Copenhagen"
i = cities.index[cities["place"] == city][0]
_, idx = finder.kneighbors(Zx[[i]])
nbrs = [j for j in idx[0] if j != i][:5]
display(cities.loc[nbrs, ["place", "alpha-2", "cost_nomad"]])

truth = cities.loc[i, "cost_nomad"]
knn_guess = cities.loc[nbrs, "cost_nomad"].mean()
mean_guess = cities.loc[cities.index != i, "cost_nomad"].mean()
print(f"{city}: actual {truth:,.0f}   neighbours {knn_guess:,.0f}   average city {mean_guess:,.0f}")

Copenhagen is much more expensive than cities that look like it on these ratings. Similarity on
safety, freedom and internet speed does not see what makes Copenhagen costly. That is the
irreducible part for this feature set: two cities with the same ratings can have very different
prices.

In [ ]:
# 2. Tree depth
rows = []
for depth in list(range(1, 13)) + [None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append({"depth": "none" if depth is None else depth,
                 "train": mean_absolute_error(y_train, m.predict(X_train)),
                 "test": mean_absolute_error(y_test, m.predict(X_test))})
depth_curve = pd.DataFrame(rows)

limited = depth_curve[depth_curve["depth"] != "none"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(limited["depth"], limited["train"], marker="o", color=BLUE, label="train")
ax.plot(limited["depth"], limited["test"], marker="o", color=ORANGE, label="test")
ax.set(xlabel="max_depth", ylabel="MAE (USD)", title="Decision tree: error against depth")
ax.legend()
plt.show()
depth_curve.set_index("depth").round(0).T

Training error falls with every extra level. Test error falls at first, then bounces around and
rises. The single split makes the curve jumpy, which is the case for exercise 3.

In [ ]:
# 3. Tune the depth with cross-validation
tree_search = GridSearchCV(DecisionTreeRegressor(random_state=0),
                           {"max_depth": list(range(1, 13)) + [None]},
                           cv=5, scoring="neg_mean_absolute_error").fit(X_train, y_train)

print("chosen depth:", tree_search.best_params_["max_depth"])
print(f"tuned tree, test MAE:    {mean_absolute_error(y_test, tree_search.predict(X_test)):,.0f}")
print(f"random forest, test MAE: {mean_absolute_error(y_test, models['random forest'].predict(X_test)):,.0f}")

A single tuned tree does not beat the forest. Averaging hundreds of deep trees removes more
variance than limiting one tree's depth. Ensembles are the start of session 10.

In [ ]:
# 4. A threshold chosen from costs, on the training data
from sklearn.model_selection import cross_val_predict

COST_MISS, COST_FALSE_ALARM = 500, 100
oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                        Xc_train, yc_train, cv=5, method="predict_proba")[:, 1]

def total_cost(truth, proba, t):
    tn, fp, fn, tp = confusion_matrix(truth, (proba >= t).astype(int)).ravel()
    return fn * COST_MISS + fp * COST_FALSE_ALARM

thresholds = np.arange(0.05, 0.96, 0.05)
costs = pd.Series([total_cost(yc_train, oof, t) for t in thresholds], index=thresholds.round(2))
best_t = costs.idxmin()
print("threshold with the lowest cost on the training folds:", best_t)

print(f"test-set cost at 0.5:           {total_cost(yc_test, proba, 0.5):,} USD")
print(f"test-set cost at {best_t}:          {total_cost(yc_test, proba, best_t):,} USD")

When a miss costs five times as much as a false alarm, the best cut-off is well below 0.5. The
model flags more cities, accepts more false alarms, and costs the agency less. The threshold was
chosen on the training folds and checked once on the test set, the same discipline as section 9.